# Image Classification with Scikit-Learn

This notebook classifies handwritten digits (0-9) using sklearn's `load_digits` dataset:
1. Explore the 8x8 pixel digit images.
2. Flatten images and train SVM and Random Forest classifiers.
3. Evaluate with confusion matrix and per-digit accuracy.
4. Analyze misclassified samples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

sns.set_style('whitegrid')
print('Imports ready.')

## 1. Load and Explore the Dataset

In [ ]:
digits = load_digits()
X = digits.data      # Flattened: (n_samples, 64)
y = digits.target     # Labels: 0-9
images = digits.images  # Original 8x8 images

print(f'Number of samples: {len(X)}')
print(f'Image shape: {images[0].shape}')
print(f'Feature vector length: {X.shape[1]}')
print(f'Classes: {np.unique(y)}')
print(f'Samples per class: {np.bincount(y)}')

In [ ]:
# Display sample digits
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    # Find two samples of each digit
    idx = np.where(y == digit)[0]
    for row in range(2):
        axes[row, digit].imshow(images[idx[row]], cmap='gray_r', interpolation='nearest')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontsize=14)

plt.suptitle('Sample Digits from the Dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Average digit images
fig, axes = plt.subplots(1, 10, figsize=(16, 2.5))
for digit in range(10):
    avg_image = images[y == digit].mean(axis=0)
    axes[digit].imshow(avg_image, cmap='hot', interpolation='nearest')
    axes[digit].set_title(str(digit), fontsize=13)
    axes[digit].axis('off')

plt.suptitle('Average Image per Digit Class', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Train-Test Split and Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

## 3. Train Classifiers

In [ ]:
# SVM classifier
svm_clf = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_clf.fit(X_train_scaled, y_train)
svm_pred = svm_clf.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f'SVM Accuracy: {svm_acc:.4f}')

# Random Forest classifier
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)  # RF doesn't need scaling
rf_pred = rf_clf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'Random Forest Accuracy: {rf_acc:.4f}')

## 4. Confusion Matrices

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for ax, pred, name, acc in [(ax1, svm_pred, 'SVM', svm_acc),
                              (ax2, rf_pred, 'Random Forest', rf_acc)]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=range(10), yticklabels=range(10))
    ax.set_title(f'{name} (Acc: {acc:.3f})')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Per-Digit Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(10)
width = 0.35

svm_per_digit = []
rf_per_digit = []
for d in range(10):
    mask = y_test == d
    svm_per_digit.append((svm_pred[mask] == y_test[mask]).mean())
    rf_per_digit.append((rf_pred[mask] == y_test[mask]).mean())

ax.bar(x - width/2, svm_per_digit, width, label=f'SVM ({svm_acc:.3f})', color='#3498db')
ax.bar(x + width/2, rf_per_digit, width, label=f'Random Forest ({rf_acc:.3f})', color='#2ecc71')

ax.set_xticks(x)
ax.set_xticklabels(range(10))
ax.set_xlabel('Digit')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.8, 1.02)
ax.set_title('Per-Digit Classification Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Detailed report for SVM
print('SVM Classification Report:')
print(classification_report(y_test, svm_pred))

## 6. Misclassification Analysis

In [ ]:
# Find misclassified samples (using SVM)
misclassified_idx = np.where(svm_pred != y_test)[0]
correct_idx = np.where(svm_pred == y_test)[0]

print(f'Misclassified: {len(misclassified_idx)} / {len(y_test)}')

# Map back to original image indices
_, test_indices = train_test_split(
    np.arange(len(X)), test_size=0.25, random_state=42, stratify=y
)

# Show misclassified samples
n_show = min(len(misclassified_idx), 16)
if n_show > 0:
    cols = min(n_show, 8)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2.5*rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_show):
        ax = axes[i // cols, i % cols]
        idx = misclassified_idx[i]
        orig_idx = test_indices[idx]
        ax.imshow(images[orig_idx], cmap='gray_r', interpolation='nearest')
        ax.set_title(f'True: {y_test[idx]}\nPred: {svm_pred[idx]}', fontsize=10, color='red')
        ax.axis('off')
    
    # Hide unused axes
    for i in range(n_show, rows * cols):
        axes[i // cols, i % cols].axis('off')
    
    plt.suptitle('Misclassified Samples (SVM)', fontsize=14, color='red')
    plt.tight_layout()
    plt.show()
else:
    print('No misclassifications! Perfect accuracy.')

In [ ]:
# Show some correctly classified samples for comparison
n_show = 16
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
np.random.seed(42)
sample_correct = np.random.choice(correct_idx, n_show, replace=False)

for i, idx in enumerate(sample_correct):
    ax = axes[i // 8, i % 8]
    orig_idx = test_indices[idx]
    ax.imshow(images[orig_idx], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'{y_test[idx]}', fontsize=12, color='green')
    ax.axis('off')

plt.suptitle('Correctly Classified Samples (SVM)', fontsize=14, color='green')
plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
# Visualize which pixels are most important for classification
importances = rf_clf.feature_importances_.reshape(8, 8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(importances, cmap='hot', interpolation='nearest')
ax1.set_title('Pixel Importance Heatmap')
ax1.axis('off')

# Overlay on average digit
avg_all = images.mean(axis=0)
ax2.imshow(avg_all, cmap='gray', interpolation='nearest', alpha=0.5)
ax2.imshow(importances, cmap='hot', interpolation='nearest', alpha=0.6)
ax2.set_title('Importance Overlaid on Average Digit')
ax2.axis('off')

plt.suptitle('Random Forest Feature Importance', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key findings:
1. The sklearn digits dataset contains 1,797 8x8 grayscale images of handwritten digits.
2. SVM (with RBF kernel) and Random Forest both achieve high accuracy on this task.
3. Misclassified digits are often ambiguous even to human readers.
4. Feature importance shows that central pixels carry the most discriminative information.
5. Certain digit pairs (e.g., 3/8, 1/7, 4/9) are more commonly confused.